In [2]:
!pip install underthesea

   ---------------------------------------- 0.0/7.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/7.3 MB ? eta -:--:--
   -------------- ------------------------- 2.6/7.3 MB 15.0 MB/s eta 0:00:01
   ---------------------------------------- 7.3/7.3 MB 24.8 MB/s  0:00:00
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 1.2/1.2 MB 31.0 MB/s  0:00:00

   -------------------- ------------------- 1/2 [underthesea]
   -------------------- ------------------- 1/2 [underthesea]
   -------------------- ------------------- 1/2 [underthesea]
   -------------------- ------------------- 1/2 [underthesea]
   -------------------- ------------------- 1/2 [underthesea]
   -------------------- ------------------- 1/2 [underthesea]
   -------------------- ------------------- 1/2 [underthesea]
   -------------------- ------------------- 1/2 [underthesea]
   -------------------- ------------------- 1/2 [underthesea]
   ------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
from datasets import load_dataset
from underthesea import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
import re

# 1. Tải dữ liệu từ Hugging Face
print("Đang tải dữ liệu...")
dataset = load_dataset("pqthinh232/vietnamese-restaurant-review-sentiment-dataset")

# Chuyển đổi các split thành DataFrame
df_train = pd.DataFrame(dataset['train'])
df_val = pd.DataFrame(dataset['validation'])
df_test = pd.DataFrame(dataset['test'])

# 2. Hàm tiền xử lý văn bản tiếng Việt
def preprocess_text(text):
    if not isinstance(text, str): return ""
    text = text.lower()
    # Loại bỏ các ký tự đặc biệt, chỉ giữ lại chữ và số
    text = re.sub(r'[^\w\s]', ' ', text)
    # Tách từ tiếng Việt (ví dụ: "nhân viên" -> "nhân_viên")
    text = word_tokenize(text, format="text")
    return text

print("Đang tiền xử lý văn bản trên cả 3 tập split...")
df_train['clean_review'] = df_train['review'].apply(preprocess_text)
df_val['clean_review'] = df_val['review'].apply(preprocess_text)
df_test['clean_review'] = df_test['review'].apply(preprocess_text)

# 3. Vector hóa TF-IDF
print("Đang chuyển đổi văn bản thành vector (TF-IDF)...")
# Ta học từ vựng (fit) trên tập Train và áp dụng (transform) cho Val/Test
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=10000)

X_train = tfidf.fit_transform(df_train['clean_review'])
X_val = tfidf.transform(df_val['clean_review'])
X_test = tfidf.transform(df_test['clean_review'])

y_train = df_train['label']
y_val = df_val['label']
y_test = df_test['label']

# 4. Huấn luyện mô hình SVM
print("Đang huấn luyện mô hình SVM...")
# class_weight='balanced' rất quan trọng vì nhãn 0 trong dữ liệu của bạn rất ít
model = SVC(kernel='linear', C=1.0, class_weight='balanced', probability=True)
model.fit(X_train, y_train)

# 5. Đánh giá mô hình trên tập Test
print("\n--- KẾT QUẢ TRÊN TẬP TEST ---")
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=['Tiêu cực (0)', 'Trung tính (1)', 'Tích cực (2)']))

# 6. Hàm dự đoán câu mới
def predict_sentiment(text):
    processed = preprocess_text(text)
    vectorized = tfidf.transform([processed])
    prediction = model.predict(vectorized)[0]
    mapping = {0: "Tiêu cực", 1: "Trung tính", 2: "Tích cực"}
    return mapping[prediction]

# Thử nghiệm thực tế
print("-" * 30)
example = "Đồ ăn ngon nhưng đợi hơi lâu, phục vụ tạm được."
print(f"Thử nghiệm: '{example}'")
print(f"Kết quả: {predict_sentiment(example)}")

PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Đang tải dữ liệu...


c:\Users\ACER\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ACER\.cache\huggingface\hub\datasets--pqthinh232--vietnamese-restaurant-review-sentiment-dataset. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 3272/3272 [00:00

Đang tiền xử lý văn bản trên cả 3 tập split...
Đang chuyển đổi văn bản thành vector (TF-IDF)...
Đang huấn luyện mô hình SVM...

--- KẾT QUẢ TRÊN TẬP TEST ---
Accuracy: 0.8289
                precision    recall  f1-score   support

  Tiêu cực (0)       0.87      0.85      0.86       622
Trung tính (1)       0.93      0.85      0.89      1981
  Tích cực (2)       0.59      0.75      0.66       669

      accuracy                           0.83      3272
     macro avg       0.80      0.82      0.80      3272
  weighted avg       0.85      0.83      0.84      3272

------------------------------
Thử nghiệm: 'Đồ ăn ngon nhưng đợi hơi lâu, phục vụ tạm được.'
Kết quả: Tích cực
